## Notebook for merging prerpcosesed datasets
### Step 1: use the post porecess scipt

In [62]:
# ==========================================
# STEP 0 — Paths & load cleaned datasets
# ==========================================
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

ROOT   = Path("/home/aidan/IMU_LM_Data")
IN_DIR = ROOT / "data" / "merged_dataset" / "per_dataset"
MERGED = ROOT / "data" / "merged_dataset"
MERGED.mkdir(parents=True, exist_ok=True)

FILES = [
    "opportunity++_postprocessed_clean.parquet",   # if you produced it
    "pamap2_postprocessed_clean.parquet",
    "recofit_postprocessed_clean.parquet",
    "samosa_postprocessed_clean.parquet",
    "ut_watch_postprocessed_clean.parquet",
    "wear_postprocessed_clean.parquet",
    "capture24_postprocessed_clean.parquet",
    "prism_postprocessed_clean.parquet",
    "shoaib_postprocessed_clean.parquet",
    "wisdm_postprocessed_clean.parquet",
]

out_path = MERGED / "IMULM_master_dataset.parquet"
if out_path.exists():
    out_path.unlink()

# canonical contract + types (pick int32 for IDs to unify)
ORDER = [
    "dataset","subject_id","session_id","timestamp_ns",
    "acc_x","acc_y","acc_z",
    "gyro_x","gyro_y","gyro_z",
    "global_activity_id","global_activity_label",
    "dataset_activity_id","dataset_activity_label",
]
TYPES = {
    "dataset": pa.string(),
    "subject_id": pa.string(),
    "session_id": pa.string(),
    "timestamp_ns": pa.int64(),
    "acc_x": pa.float32(),
    "acc_y": pa.float32(),
    "acc_z": pa.float32(),
    "gyro_x": pa.float32(),
    "gyro_y": pa.float32(),
    "gyro_z": pa.float32(),
    "global_activity_id": pa.int32(),       # <- unify to int32 too (optional, but consistent)
    "global_activity_label": pa.string(),
    "dataset_activity_id": pa.int32(),      # <- THIS fixes your crash
    "dataset_activity_label": pa.string(),
}
CANON_SCHEMA = pa.schema([(c, TYPES[c]) for c in ORDER])

writer = None
total = 0

for fname in FILES:
    p = IN_DIR / fname
    if not p.exists():
        print(f"[SKIP] missing: {p}")
        continue

    pf = pq.ParquetFile(p)
    print(f"[ADD] {fname:35s} rows={pf.metadata.num_rows:,} row_groups={pf.num_row_groups}")

    for batch in pf.iter_batches(batch_size=1_000_000):
        tbl = pa.Table.from_batches([batch])

        # ensure column order + cast types (only for columns that exist)
        cols = []
        for c in ORDER:
            arr = tbl[c]
            if arr.type != TYPES[c]:
                arr = pc.cast(arr, TYPES[c])
            cols.append(arr)
        tbl = pa.Table.from_arrays(cols, names=ORDER).cast(CANON_SCHEMA)

        if writer is None:
            writer = pq.ParquetWriter(out_path, CANON_SCHEMA, compression="zstd")

        writer.write_table(tbl)
        total += tbl.num_rows

if writer is not None:
    writer.close()

print(f"\nWrote master: {out_path} | rows={total:,}")


[ADD] opportunity++_postprocessed_clean.parquet rows=985,421 row_groups=2
[ADD] pamap2_postprocessed_clean.parquet  rows=1,271,041 row_groups=2
[ADD] recofit_postprocessed_clean.parquet rows=7,745,018 row_groups=8
[ADD] samosa_postprocessed_clean.parquet  rows=1,204,655 row_groups=2
[ADD] ut_watch_postprocessed_clean.parquet rows=8,146,709 row_groups=9
[ADD] wear_postprocessed_clean.parquet    rows=3,462,973 row_groups=4
[ADD] capture24_postprocessed_clean.parquet rows=699,011,946 row_groups=700
[ADD] prism_postprocessed_clean.parquet   rows=3,037,272 row_groups=4
[ADD] shoaib_postprocessed_clean.parquet  rows=1,170,000 row_groups=2
[ADD] wisdm_postprocessed_clean.parquet   rows=8,045,562 row_groups=9

Wrote master: /home/aidan/IMU_LM_Data/data/merged_dataset/IMULM_master_dataset.parquet | rows=734,080,597


In [61]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from pathlib import Path

# P = "/home/aidan/IMU_LM_Data/data/merged_dataset/master_dataset.parquet"

DATASET = "samosa"   # <-- change per dataset
CLEAN   =    # <-- False=audit only, True=drop BAD RUN rows

OUT_DIR = Path("/home/aidan/IMU_LM_Data/data/merged_dataset/per_dataset")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_P = OUT_DIR / f"{DATASET}.parquet" if not CLEAN else OUT_DIR / f"{DATASET}_postprocessed_clean.parquet"

GKEY  = ["dataset","subject_id","session_id"]
TS    = "timestamp_ns"
LABEL = "dataset_activity_label"

MIN_SAMPLES = 128
MIN_DUR_NS  = int(2.56 * 1e9)

DT_NS  = int(1e9 / 50)   # 20,000,000
TOL_NS = 2_000_000       # ±2ms
MAX_DT = DT_NS + TOL_NS
MIN_DT = DT_NS - TOL_NS

# pf = pq.ParquetFile(P)

# state[key] = (last_lab, run_len, t0, last_ts)
state = {}

bad_runs = total_runs = 0
min_run_len = None
min_run_dur = None
bad_steps = 0
removed_rows = 0

def finish(run_len, t0, t1):
    global bad_runs, total_runs, min_run_len, min_run_dur
    dur = int(t1 - t0)
    total_runs += 1
    min_run_len = run_len if min_run_len is None else min(min_run_len, run_len)
    min_run_dur = dur     if min_run_dur is None else min(min_run_dur, dur)
    if run_len < MIN_SAMPLES or dur < MIN_DUR_NS:
        bad_runs += 1

writer = None

for rg in range(pf.num_row_groups):
    df = pf.read_row_group(rg).to_pandas()
    df = df[df["dataset"].astype(str) == DATASET]
    if df.empty:
        continue
    df = df.sort_values(GKEY+[TS], kind="mergesort")

    # CLEAN: drop rows that belong to BAD RUNS (by MIN_SAMPLES/MIN_DUR)
    if CLEAN:
        keep = np.ones(len(df), dtype=bool)
        df_idx = df.index.to_numpy()

        for key, g in df.groupby(GKEY, sort=False):
            idx = g.index.to_numpy()
            lab = g[LABEL].to_numpy()
            ts  = g[TS].to_numpy(np.int64)

            run_start = 0
            last_lab = lab[0]

            for i in range(1, len(g)):
                if lab[i] != last_lab:
                    run_len = i - run_start
                    dur = int(ts[i-1] - ts[run_start])
                    if run_len < MIN_SAMPLES or dur < MIN_DUR_NS:
                        keep[np.isin(df_idx, idx[run_start:i])] = False
                    run_start = i
                    last_lab = lab[i]

            run_len = len(g) - run_start
            dur = int(ts[-1] - ts[run_start])
            if run_len < MIN_SAMPLES or dur < MIN_DUR_NS:
                keep[np.isin(df_idx, idx[run_start:])] = False

        removed_rows += int((~keep).sum())
        df = df.loc[df.index[keep]]

    # ALWAYS audit on whatever df we're writing (raw or cleaned)
    for key, g in df.groupby(GKEY, sort=False):
        lab = g[LABEL].to_numpy()
        ts  = g[TS].to_numpy(np.int64)

        if key not in state:
            state[key] = (lab[0], 1, ts[0], ts[0])
            start = 1
        else:
            start = 0

        last_lab, run_len, t0, last_ts = state[key]
        for i in range(start, len(g)):
            if lab[i] == last_lab:
                dt = int(ts[i] - last_ts)
                if dt <= 0 or dt < MIN_DT or dt > MAX_DT:
                    bad_steps += 1
                run_len += 1
                last_ts = ts[i]
            else:
                finish(run_len, t0, last_ts)
                last_lab = lab[i]
                run_len = 1
                t0 = ts[i]
                last_ts = ts[i]
        state[key] = (last_lab, run_len, t0, last_ts)

    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(str(OUT_P), table.schema)
    writer.write_table(table)

for (last_lab, run_len, t0, last_ts) in state.values():
    finish(run_len, t0, last_ts)

if writer is not None:
    writer.close()

print(f"{DATASET} in master_dataset:",
      "PASS" if (bad_runs==0) else "FAIL",
      "| mode=", "CLEAN" if CLEAN else "AUDIT",
      "| out=", str(OUT_P),
      "| total_runs=", total_runs,
      "| bad_runs=", bad_runs,
      "| bad_steps=", bad_steps,
      "| removed_rows=", removed_rows if CLEAN else 0,
      "| min_run_len=", min_run_len,
      "| min_run_dur_s=", None if min_run_dur is None else round(min_run_dur/1e9,3))


capture24 in master_dataset: PASS | mode= CLEAN | out= /home/aidan/IMU_LM_Data/data/merged_dataset/per_dataset/capture24_postprocessed_clean.parquet | total_runs= 0 | bad_runs= 0 | bad_steps= 0 | removed_rows= 0 | min_run_len= None | min_run_dur_s= None


In [ ]:
# # aduiot for the step break

# import pyarrow.parquet as pq
# import pandas as pd
# import numpy as np

# P = "/home/aidan/IMU_LM_Data/data/merged_dataset/per_dataset/pamap2_postprocessed_clean.parquet"

# GKEY  = ["dataset","subject_id","session_id"]
# TS    = "timestamp_ns"
# LABEL = "dataset_activity_label"

# DT_NS  = int(1e9 / 50)
# TOL_NS = 2_000_000
# MAX_DT = DT_NS + TOL_NS
# MIN_DT = DT_NS - TOL_NS

# pf = pq.ParquetFile(P)

# bad_le0 = 0
# bad_small = 0
# bad_big = 0
# dt_min = None
# dt_max = None
# dt_sum = 0
# dt_n = 0

# # optional: track worst offenders by session
# worst = {}  # key -> bad_count

# for rg in range(pf.num_row_groups):
#     df = pf.read_row_group(rg, columns=GKEY+[TS,LABEL]).to_pandas()
#     if df.empty:
#         continue
#     df = df.sort_values(GKEY+[TS], kind="mergesort")

#     for key, g in df.groupby(GKEY, sort=False):
#         ts  = g[TS].to_numpy(np.int64)
#         lab = g[LABEL].to_numpy()

#         # only inside same-label runs
#         for i in range(1, len(g)):
#             if lab[i] != lab[i-1]:
#                 continue
#             dt = int(ts[i] - ts[i-1])

#             # global dt stats for "normal" steps (dt>0)
#             if dt > 0:
#                 dt_sum += dt
#                 dt_n += 1
#                 dt_min = dt if dt_min is None else min(dt_min, dt)
#                 dt_max = dt if dt_max is None else max(dt_max, dt)

#             # classify bad steps
#             if dt <= 0:
#                 bad_le0 += 1
#                 worst[key] = worst.get(key, 0) + 1
#             elif dt < MIN_DT:
#                 bad_small += 1
#                 worst[key] = worst.get(key, 0) + 1
#             elif dt > MAX_DT:
#                 bad_big += 1
#                 worst[key] = worst.get(key, 0) + 1

# print("dt summary (within same-label adjacencies):",
#       "mean_dt_ms=", None if dt_n==0 else round((dt_sum/dt_n)/1e6, 3),
#       "| min_dt_ms=", None if dt_min is None else round(dt_min/1e6, 3),
#       "| max_dt_ms=", None if dt_max is None else round(dt_max/1e6, 3))

# print("bad_steps breakdown:",
#       "| dt<=0 =", bad_le0,
#       "| dt<MIN =", bad_small,
#       "| dt>MAX =", bad_big,
#       "| total =", bad_le0 + bad_small + bad_big)

# # top 10 worst sessions
# top = sorted(worst.items(), key=lambda x: x[1], reverse=True)[:10]
# print("\nworst sessions (top 10):")
# for k, c in top:
#     print(k, "bad_steps=", c)


In [60]:
# import pyarrow.parquet as pq
# import numpy as np
# import pandas as pd
# from collections import Counter

# P = "/home/aidan/IMU_LM_Data/data/merged_dataset/master_dataset.parquet"
# DATASET = "capture24"

# GKEY  = ["dataset","subject_id","session_id"]
# LABEL = "dataset_activity_label"
# TS    = "timestamp_ns"

# pf = pq.ParquetFile(P)

# run_lengths = []

# for rg in range(pf.num_row_groups):
#     df = pf.read_row_group(rg, columns=GKEY+[LABEL, TS]).to_pandas()
#     df = df[df["dataset"].astype(str) == DATASET]
#     if df.empty:
#         continue
#     df = df.sort_values(GKEY+[TS], kind="mergesort")

#     for _, g in df.groupby(GKEY, sort=False):
#         lab = g[LABEL].to_numpy()

#         run_len = 1
#         for i in range(1, len(lab)):
#             if lab[i] == lab[i-1]:
#                 run_len += 1
#             else:
#                 run_lengths.append(run_len)
#                 run_len = 1
#         run_lengths.append(run_len)

# # stats
# cnt = Counter(run_lengths)

# total_runs = sum(cnt.values())
# singleton_runs = cnt.get(1, 0)
# short_runs = sum(v for k, v in cnt.items() if k < 128)

# print("Opportunity++ run length audit")
# print("--------------------------------")
# print("total runs        :", total_runs)
# print("singleton runs    :", singleton_runs, f"({singleton_runs/total_runs:.1%})")
# print("runs <128 samples :", short_runs, f"({short_runs/total_runs:.1%})")
# print("\nTop run lengths:")
# for k in sorted(cnt)[:10]:
#     print(f"  len={k:<4} count={cnt[k]}")


Opportunity++ run length audit
--------------------------------
total runs        : 27058
singleton runs    : 18 (0.1%)
runs <128 samples : 192 (0.7%)

Top run lengths:
  len=1    count=18
  len=2    count=6
  len=3    count=3
  len=4    count=4
  len=5    count=1
  len=6    count=2
  len=7    count=4
  len=8    count=3
  len=9    count=2
  len=11   count=5


In [1]:
# ==========================================
# STEP 0 — Paths & load cleaned datasets
# ==========================================
from pathlib import Path
import pandas as pd

ROOT    = Path("/home/aidan/IMU_LM_Data")
CLEANED = ROOT / "data" / "cleaned_premerge"
MERGED  = ROOT / "data" / "merged_dataset"

print("CLEANED dir :", CLEANED)
print("MERGED  dir :", MERGED)

# Cleaned parquet files we expect (only load ones that actually exist)
DATASETS = {
    "opportunity": "opportunity_clean_data.parquet",
    "pamap2":      "pamap2_clean_data.parquet",
    "recofit":     "recofit_clean_data.parquet",
    "samosa":      "samosa_clean_data.parquet",
    "ut_watch":    "ut_watch_clean_data.parquet",
    "wear":        "wear_clean_data.parquet",
    "capture24":   "capture24_clean_data.parquet",
    "prism":       "prism_clean_data.parquet",
    "shoaib":      "shoaib_clean_data.parquet",
    "wisdm":       "wisdm_clean_data.parquet",
}

loaded = {}
for ds_name, fname in DATASETS.items():
    path = CLEANED / fname
    if not path.exists():
        print(f"[WARN] Missing cleaned file for {ds_name}: {path}")
        continue

    df = pd.read_parquet(path)
    print(f"[OK] Loaded {ds_name:10s}: {len(df):,} rows")
    loaded[ds_name] = df

if not loaded:
    raise SystemExit("No cleaned datasets loaded — check CLEANED path / filenames.")

print("\nLoaded datasets:", list(loaded.keys()))



CLEANED dir : /home/aidan/IMU_LM_Data/data/cleaned_premerge
MERGED  dir : /home/aidan/IMU_LM_Data/data/merged_dataset
[OK] Loaded opportunity: 1,215,455 rows
[OK] Loaded pamap2    : 1,271,148 rows
[OK] Loaded recofit   : 7,751,906 rows
[OK] Loaded samosa    : 1,205,141 rows
[OK] Loaded ut_watch  : 8,146,709 rows
[OK] Loaded wear      : 3,466,400 rows


: 

### Step 1: Create Master Dataset

In [2]:
# ==========================================
# STEP 1 — Merge, sanity checks, save
# ==========================================

# 1) Concatenate (simple stack)
unified = pd.concat(
    [loaded[k] for k in sorted(loaded.keys())],
    ignore_index=True
)

# 2) Sort by core index columns
unified = unified.sort_values(
    ["dataset", "subject_id", "session_id", "timestamp_ns"]
).reset_index(drop=True)

print(f"\nUNIFIED rows: {len(unified):,}")

# 3) Basic stats
print("\nRows per dataset:")
print(unified["dataset"].value_counts())

print("\nSubjects per dataset:")
print(unified.groupby("dataset")["subject_id"].nunique())

print("\nSessions per dataset:")
print(unified.groupby("dataset")["session_id"].nunique())

# 4) Required-not-null coverage (hard-coded from schema)
REQ_NOT_NULL = [
    "dataset", "subject_id", "session_id", "timestamp_ns",
    "acc_x", "acc_y", "acc_z",
    "global_activity_id", "global_activity_label",
    "dataset_activity_id", "dataset_activity_label",
]
pct_req = unified[REQ_NOT_NULL].notnull().all(axis=1).mean() * 100
print(f"\nRows meeting required-not-null contract: {pct_req:.2f}%")

# 5) Global mapping coverage + label distribution
UNKNOWN_ID = 9000
cov = (unified["global_activity_id"] != UNKNOWN_ID).mean() * 100
print(f"Global activity mapping coverage: {cov:.1f}% (unknown={UNKNOWN_ID})")

print("\nTop-20 canonical labels (global_activity_label):")
print(unified["global_activity_label"].value_counts().head(20))

# 6) Quick 1–1 check: global ID → label
pairs = unified[["global_activity_id", "global_activity_label"]].drop_duplicates()
id_to_lbl_counts = pairs.groupby("global_activity_id")["global_activity_label"].nunique()
print("\nGlobal id→label one-to-one:", bool((id_to_lbl_counts == 1).all()))

# 7) Save unified parquet
MERGED.mkdir(parents=True, exist_ok=True)
out_path = MERGED / "unified_dataset.parquet"
unified.to_parquet(out_path, index=False)
print("\nSaved unified dataset to:", out_path)
print("Done.")


UNIFIED rows: 23,056,759

Rows per dataset:
dataset
ut_watch         8146709
recofit          7751906
wear             3466400
pamap2           1271148
opportunity++    1215455
samosa           1205141
Name: count, dtype: int64

Subjects per dataset:
dataset
opportunity++     4
pamap2            9
recofit          94
samosa           20
ut_watch         20
wear             24
Name: subject_id, dtype: int64

Sessions per dataset:
dataset
opportunity++      6
pamap2             2
recofit           73
samosa           125
ut_watch           4
wear               1
Name: session_id, dtype: int64

Rows meeting required-not-null contract: 100.00%
Global activity mapping coverage: 92.6% (unknown=9000)

Top-20 canonical labels (global_activity_label):
global_activity_label
rest_inactive            7224537
other                    1805574
exercise_upper           1765088
stretching               1695420
device_offbody           1301735
exercise_core            1165114
walk                     1